# Masked DFM on K562: training and sampling

## Run the following cell only if running on colab

In [5]:
!git clone -b experimental https://github.com/darreld081/CIS6270-Final-Project.git
%cd CIS6270-Final-Project
!pip install -q pytorch-lightning
import sys; sys.path.insert(0, "src")

Cloning into 'CIS6270-Final-Project'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 56 (delta 13), reused 45 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 5.26 MiB | 21.13 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/home/dylon/CIS6270-Final-Project/CIS6270-Final-Project


## Run the following cell if running locally

In [1]:
import sys; sys.path.insert(0, "src")


### Imports

In [2]:
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from dfm.module import MaskedDFM
from datetime import datetime

### CUDA check

In [3]:
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")

CUDA available: True
CUDA version: 13.2
Device count: 1
Device 0: NVIDIA GeForce RTX 5060


### Generate data splits

In [4]:
import os
os.getcwd()


'/home/dylon/CIS6270-Final-Project'

In [5]:
# Preprocess: download raw DHS table, keep K562, encode to int8[N, 200]
from dfm.data import download, load_table, build_splits

splits = build_splits(load_table(download("data/raw")))
splits["train"].shape

(9769, 200)

### Training loop

In [ ]:
# Train
EPOCHS = 200
use_checkpoint = False
pl.seed_everything(0)
train_loader = DataLoader(torch.from_numpy(splits["train"]).long(),
                          batch_size=256, shuffle=True, drop_last=True, num_workers=2)
module = MaskedDFM()
trainer = pl.Trainer(max_epochs=EPOCHS, gradient_clip_val=1.0,
                     precision="bf16-mixed" if torch.cuda.is_available() else "32-true",
                     logger=False, enable_checkpointing=False)
trainer.fit(module, train_loader)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
trainer.save_checkpoint(f"checkpoints/base_dfm_{timestamp}.ckpt")

Seed set to 0
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 5060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/home/dylon/CIS6270-Final-Project/.venv/lib/python3.14/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecat

Epoch 26:  18%|█▊        | 7/38 [00:00<00:04,  7.04it/s, train_loss_step=1.320, train_loss_epoch=1.310] 


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/dylon/CIS6270-Final-Project/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Run to load checkpoint (optional)

In [6]:
use_checkpoint = True

### Single sampler for testing

In [51]:
# Sample one sequence
from dfm.alphabet import decode
from dfm.sampler import sample

device = "cuda" if torch.cuda.is_available() else "cpu"
if use_checkpoint:
    model = MaskedDFM.load_from_checkpoint("checkpoints/base_dfm_20261127_020259.ckpt", map_location=device)
else:
    model = module.model.to(device).eval()
seq = sample(model, num_samples=1, num_steps=128, device=device, seed=0)
print(decode(seq[0]))

AGCAGTTACGGTGATATAAGCCTCCACATGTGGCTCCTTAGGCCTCAAGTCTAATCTCTGTGCAAGGCTGTTACCTTCTGCTTTTGCTTTCCTACTGCAGAAACATGCTACTTATTTACAGTGACAACCATGTGTCAGCTTCTCCGAGCCCTTGGCTGCCTTACCTCCTTTGATACTGCCTTGCAGGGAGGGTGGGCCAC


### Run some metrics to verify performance

In [12]:
# Evaluate 1024 samples against held-out test sequences (chr1).
# Good: k-mer JSD well below the uniform baseline, GC mean close to test and std ratio near 1,
# unique near 1, memorized near 0, pairwise Hamming close to test (much lower = mode collapse).
from dfm.metrics import evaluate

gen = sample(model, num_samples=1024, num_steps=128, batch_size=256, device=device, seed=1)
r = evaluate(gen, splits["test"], train=splits["train"])
print(f"GC mean   gen {r['gc_generated_mean']:.3f}  test {r['gc_reference_mean']:.3f}  std ratio {r['gc_std_ratio']:.2f}")
for k in r["kmer_jsd"]:
    print(f"{k}-mer JSD  gen {r['kmer_jsd'][k]:.4f}  uniform {r['kmer_jsd_uniform_baseline'][k]:.4f}")
print(f"unique {r['unique_fraction']:.3f}  memorized {r['memorization_fraction']:.3f}")
print(f"pairwise Hamming  gen {r['mean_pairwise_hamming']:.3f}  test {r['reference_pairwise_hamming']:.3f}")

GC mean   gen 0.489  test 0.492  std ratio 0.89
3-mer JSD  gen 0.0008  uniform 0.0363
4-mer JSD  gen 0.0017  uniform 0.0575
6-mer JSD  gen 0.0105  uniform 0.1061
unique 1.000  memorized 0.000
pairwise Hamming  gen 0.750  test 0.750


### Fréchet Biological Distance (FBD)
Embed sequences with a CNN trained to classify the 4 DHS cell lines, then compute the Fréchet distance between generated and held-out K562 embeddings (FID for DNA, as in Stark et al. 2024). **Lower is better.** Read it against the floor (real K562 val sequences, unseen by the classifier, vs test) and ceiling (uniform random vs test), all at the same N, since FD is biased upward at small N.

In [18]:
# Train (or load cached) the cell-type classifier used as the FBD embedder.
# Same chr1/chr2 held-out split, so K562 test sequences are never seen in training.
from pathlib import Path
from dfm.data import build_labeled_splits
from dfm.fbd import train_classifier, save_classifier, load_classifier

FBD_CKPT = Path("checkpoints/fbd_classifier.pt")
if FBD_CKPT.exists():
    fbd_model, fbd_metrics = load_classifier(FBD_CKPT, device=device)
else:
    labeled = build_labeled_splits(load_table(download("data/raw")))
    fbd_model, fbd_metrics = train_classifier(labeled, device=device)
    save_classifier(fbd_model, FBD_CKPT, fbd_metrics)
print(f"cell-type classifier  val acc {fbd_metrics['val_acc']:.3f}  "
      f"test acc {fbd_metrics['test_acc']:.3f}  (chance {fbd_metrics['chance']:.3f})")

cell-type classifier  val acc 0.780  test acc 0.766  (chance 0.289)


In [19]:
# FBD of the generated samples vs held-out K562 test, bracketed by a real-data floor
# and a uniform-random ceiling at the same N. Good: gen close to the floor, far below the ceiling.
# The floor must be real sequences the classifier never trained on (K562 val, chr2): training
# sequences embed differently from unseen ones and give a falsely high floor (~1.0 vs ~0.07).
import numpy as np
from dfm.fbd import fbd
from dfm.metrics import uniform_random_sequences

n = min(len(gen), len(splits["val"]))
rng = np.random.default_rng(0)
gen_n = gen[rng.choice(len(gen), n, replace=False)]
real = splits["val"][rng.choice(len(splits["val"]), n, replace=False)]
rand = uniform_random_sequences(n, splits["test"].shape[1], rng)

fbd_scores = {
    "generated": fbd(fbd_model, gen_n, splits["test"], device),
    "K562 val (floor)": fbd(fbd_model, real, splits["test"], device),
    "uniform random (ceiling)": fbd(fbd_model, rand, splits["test"], device),
}
print(f"N = {n} per set, reference = K562 test ({len(splits['test'])})")
for name, score in fbd_scores.items():
    print(f"FBD  {name:<26} {score:8.4f}")

N = 878 per set, reference = K562 test (1321)
FBD  generated                    0.4550
FBD  K562 val (floor)             0.0690
FBD  uniform random (ceiling)     4.7694
